In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d96d2596-5de0-4ebb-8679-543ad32c3a6b;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 402ms :: artifacts dl 16ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
df_sellers = spark.read.option('delimiter', ',') \
              .option('header', 'true') \
              .option('nullValue', 'NULL') \
              .csv('s3a://last-mile-optimization-raw/dataset-orders/olist_sellers_dataset.csv')
df_sellers.show()

26/04/04 22:15:59 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                 04195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
|c240c4061717ac180...|                 20920|   rio de janeiro|          RJ|
|e49c26c3edfa46d22...|                 55325|           brejao|          PE|
|1b938a7ec6ac5061a...|                 16304|        penapolis|          SP|
|768a86e36ad6aae3d...|                 01529|        sao paulo|          SP|
|ccc4bbb5f32a6ab2b...|                 80310|         curitiba|          PR|

In [4]:
from pyspark.sql.functions import col, upper, regexp_replace, when, split
from pyspark.sql.types import StringType

In [5]:
# Converter 'seller_zip_code_prefix' para String
df_sellers = df_sellers.withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast(StringType()))

# Filtrar por estado 'SP'
df_sellers_sp = df_sellers.filter(col("seller_state") == "SP")
df_sellers_sp.show()

+--------------------+----------------------+--------------------+------------+
|           seller_id|seller_zip_code_prefix|         seller_city|seller_state|
+--------------------+----------------------+--------------------+------------+
|3442f8959a84dea7e...|                 13023|            campinas|          SP|
|d1b65fc7debc3361e...|                 13844|          mogi guacu|          SP|
|c0f3eea2e14555b6f...|                 04195|           sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|   braganca paulista|          SP|
|1b938a7ec6ac5061a...|                 16304|           penapolis|          SP|
|768a86e36ad6aae3d...|                 01529|           sao paulo|          SP|
|a7a9b880c49781da6...|                 13530|           itirapina|          SP|
|8bd0f31cf0a614c65...|                 01222|           sao paulo|          SP|
|05a48cc8859962767...|                 05372|           sao paulo|          SP|
|f9ec7093df3a7b346...|                 0

In [6]:
# Colocar nomes de cidade em letra maiúscula 
df_sellers_sp = df_sellers_sp.withColumn('seller_city', upper(col('seller_city')))

# Remover acentos
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), '[áàâãä]', 'a'))
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), '[éèêë]', 'e'))
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), '[íìîï]', 'i'))
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), '[óòôõö]', 'o'))
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), '[úùûü]', 'u'))
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), 'ç', 'c'))

# Remover a coluna customer_state
df_sellers_sp = df_sellers_sp.drop("seller_state")

df_sellers_sp.show()

+--------------------+----------------------+--------------------+
|           seller_id|seller_zip_code_prefix|         seller_city|
+--------------------+----------------------+--------------------+
|3442f8959a84dea7e...|                 13023|            CAMPINAS|
|d1b65fc7debc3361e...|                 13844|          MOGI GUACU|
|c0f3eea2e14555b6f...|                 04195|           SAO PAULO|
|51a04a8a6bdcb23de...|                 12914|   BRAGANCA PAULISTA|
|1b938a7ec6ac5061a...|                 16304|           PENAPOLIS|
|768a86e36ad6aae3d...|                 01529|           SAO PAULO|
|a7a9b880c49781da6...|                 13530|           ITIRAPINA|
|8bd0f31cf0a614c65...|                 01222|           SAO PAULO|
|05a48cc8859962767...|                 05372|           SAO PAULO|
|f9ec7093df3a7b346...|                 05138|           SAO PAULO|
|4e6015589b781adaa...|                 11440|             GUARUJA|
|4cf490a58259286ad...|                 14910|           TABATI

In [7]:
# Para detectar e corrigir a cidade "jacarei / sao paulo" para apenas "jacarei", podemos usar a função `when` para 
# aplicar uma condição. Se a coluna `seller_city` contiver "/", pegamos a parte antes da barra, caso contrário, 
# mantemos o nome original. Isso é eficiente mesmo para milhões de linhas.
df_sellers_sp = df_sellers_sp.withColumn('seller_city', 
    when(col('seller_city').contains('/'), split(col('seller_city'), ' / ').getItem(0))
    .otherwise(col('seller_city'))
)

df_sellers_sp.show()


+--------------------+----------------------+--------------------+
|           seller_id|seller_zip_code_prefix|         seller_city|
+--------------------+----------------------+--------------------+
|3442f8959a84dea7e...|                 13023|            CAMPINAS|
|d1b65fc7debc3361e...|                 13844|          MOGI GUACU|
|c0f3eea2e14555b6f...|                 04195|           SAO PAULO|
|51a04a8a6bdcb23de...|                 12914|   BRAGANCA PAULISTA|
|1b938a7ec6ac5061a...|                 16304|           PENAPOLIS|
|768a86e36ad6aae3d...|                 01529|           SAO PAULO|
|a7a9b880c49781da6...|                 13530|           ITIRAPINA|
|8bd0f31cf0a614c65...|                 01222|           SAO PAULO|
|05a48cc8859962767...|                 05372|           SAO PAULO|
|f9ec7093df3a7b346...|                 05138|           SAO PAULO|
|4e6015589b781adaa...|                 11440|             GUARUJA|
|4cf490a58259286ad...|                 14910|           TABATI

In [8]:
df_sellers_sp.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted/dataset-orders/sellers_cleaned_dataset.csv')

spark.stop()

26/04/04 22:16:12 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/04 22:16:12 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
